## Support-Intent Dataset Generator

Builds a synthetic **utterance → intent** CSV for customer support. Choose a model, set context and row count, then generate; output streams in the UI and the CSV is downloadable.

**What’s inside**
- 4-bit quantized Hugging Face models (e.g. Llama 3.1 8B, Qwen2.5 7B)
- Thread-safe streaming with error handling (no UI hang on failures)
- Gradio UI: model dropdown, Load, Generate, Download CSV

**Open in Colab:** [Launch in Google Colab](https://colab.research.google.com/drive/1xajKX6QSamSop3FaXYvOMv_NRkl1699d#scrollTo=LJMxF1v-Wqmx)

*If model load fails: set **HF_TOKEN** (Colab Secrets or `.env`), accept the model license on Hugging Face (e.g. Meta-Llama), and use a GPU runtime in Colab.*

In [ ]:
!pip install -q torch bitsandbytes transformers accelerate gradio python-dotenv

In [ ]:
MODEL_OPTIONS = {
    "Llama 3.1 8B": "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "Qwen2.5 7B": "Qwen/Qwen2.5-1.5B-Instruct",
}

In [ ]:
import os
import re
import tempfile
import traceback
from threading import Thread

import torch
import gradio as gr
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextIteratorStreamer

In [ ]:
# HF token: Colab userdata or local .env
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
def create_model(mid, token=None):
    if not (mid and str(mid).strip()):
        raise ValueError("Model id is empty. Use a valid Hugging Face model id.")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True, token=token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mod = AutoModelForCausalLM.from_pretrained(mid, quantization_config=quant_config, token=token, device_map="auto")
    return tok, mod

In [ ]:
tokenizer, model = create_model("meta-llama/Meta-Llama-3.1-8B-Instruct", token=hf_token)

In [ ]:
SUPPORT_INTENT_TEMPLATE = {
    "system_prompt": "You output only a CSV with exactly two columns: utterance,intent. No header explanation. Use intents like: billing, cancellation, technical_issue, account_access, general_inquiry, refund, other. One customer support utterance per row.",
    "default_n_rows": 20,
}

def build_support_intent_prompt(context=None, n_rows=20):
    focus = f"Focus on: {context}." if context else "Customer support."
    return f"{focus} Generate {n_rows} rows of utterance,intent."

In [ ]:
def create_inputs(tokenizer, history, prompt, system_prompt=None):
    sp = system_prompt or "You are a helpful assistant."
    messages = [{"role": "system", "content": sp}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": prompt})
    out = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
    if hasattr(out, "input_ids"):
        out = out["input_ids"]
    elif isinstance(out, dict) and "input_ids" in out:
        out = out["input_ids"]
    return out

In [ ]:
def generate_support_intent(context, n_rows):
    prompt = build_support_intent_prompt(context or "", n_rows)
    input_ids = create_inputs(tokenizer, [], prompt, system_prompt=SUPPORT_INTENT_TEMPLATE["system_prompt"])
    param = next(model.parameters(), None)
    if param is not None:
        input_ids = input_ids.to(param.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True, decode_kwargs={"skip_special_tokens": True})
    exc_info = []

    def generate_with_catch():
        try:
            model.generate(
                input_ids=input_ids,
                max_new_tokens=512,
                streamer=streamer,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        except Exception as e:
            exc_info.append((str(e), traceback.format_exc()))
        finally:
            streamer.end()

    thread = Thread(target=generate_with_catch)
    thread.start()
    full = ""
    for chunk in streamer:
        full += chunk.replace("<|eot_id|>", "")
        yield full
    if exc_info:
        err_msg, tb = exc_info[0]
        full += f"\n\n[SYSTEM ERROR]: {err_msg}\n\n--- Traceback ---\n{tb}"
        yield full

In [ ]:
def parse_csv_from_response(text):
    text = re.sub(r"```\w*\n?", "", text).strip()
    lines = [line for line in text.splitlines() if "," in line]
    return "\n".join(lines) if lines else ""

In [ ]:
def load_model(choice):
    global tokenizer, model
    model_id = MODEL_OPTIONS.get(choice) if choice else None
    if not model_id or not model_id.strip():
        return "Select a model from the dropdown first."
    tokenizer, model = create_model(model_id, token=hf_token)
    return f"Loaded {choice}"

def stream_run_ui(context, n_rows):
    n = max(5, min(50, int(n_rows) if n_rows else 20))
    for out_text in generate_support_intent(context, n):
        yield out_text, ""
    csv = parse_csv_from_response(out_text)
    yield out_text, csv

def make_csv_download(csv_text):
    if not (csv_text and csv_text.strip()):
        return None
    fd, path = tempfile.mkstemp(suffix=".csv")
    try:
        with os.fdopen(fd, "w") as f:
            f.write(csv_text)
        return path
    except Exception:
        os.close(fd)
        raise

with gr.Blocks() as demo:
    gr.Markdown("Support intent dataset")
    model_dd = gr.Dropdown(choices=list(MODEL_OPTIONS.keys()), value=list(MODEL_OPTIONS.keys())[0], label="Model")
    load_btn = gr.Button("Load model")
    status = gr.Textbox(label="Status", interactive=False)
    load_btn.click(fn=load_model, inputs=[model_dd], outputs=[status])
    ctx = gr.Textbox(label="Context (optional)", placeholder="e.g. SaaS billing")
    rows = gr.Number(label="Rows", value=20, minimum=5, maximum=50)
    gen_btn = gr.Button("Generate")
    out_text = gr.Textbox(label="Output", lines=8)
    out_csv = gr.Textbox(label="CSV", lines=8)
    download_btn = gr.Button("Download CSV")
    csv_file = gr.File(label="Download", visible=True)
    gen_btn.click(fn=stream_run_ui, inputs=[ctx, rows], outputs=[out_text, out_csv])
    download_btn.click(fn=make_csv_download, inputs=[out_csv], outputs=[csv_file])

demo.launch()